## Semantic Search Ready NLP Pipeline (End-to-End)

This notebook builds a complete NLP preprocessing and semantic similarity pipeline from raw text using:

NLTK (stopwords, basic tokenization utilities)
spaCy (lemmatization)
scikit-learn (BoW/TF‑IDF + cosine similarity)
Gensim (Word2Vec + Doc2Vec sentence embeddings)

In [ ]:
# Install and import dependencies
# Download language models/resources

!pip -q install datasets spacy gensim wordcloud umap-learn
!python -m spacy download en_core_web_sm -q

import re
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from datasets import load_dataset

import nltk
nltk.download("stopwords")
nltk.download("punkt")

from nltk.corpus import stopwords

import spacy
nlp = spacy.load("en_core_web_sm")

from collections import Counter

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from gensim.models import Word2Vec
from gensim.models.doc2vec import Doc2Vec, TaggedDocument

from wordcloud import WordCloud
import umap

## Step 1 — Load raw text data (HuggingFace Dataset)

We load a HuggingFace datasets both for reviews and comments and keep a manageable sample size for Colab runtime.

In [ ]:
# Load Dataset (HuggingFace) — Merge Reviews + Comments
# Reviews: yelp_polarity
# Comments: civil_comments
# Creates a single corpus with a 'source' column

from datasets import load_dataset
import pandas as pd

# Reviews dataset
reviews_ds = load_dataset("yelp_polarity")["train"]
df_reviews = pd.DataFrame(reviews_ds).sample(n=4000, random_state=42).reset_index(drop=True)
df_reviews = df_reviews.rename(columns={"text": "text"})
df_reviews["source"] = "review"
df_reviews["label"] = df_reviews["label"]

# Comments dataset
comments_ds_all = load_dataset("civil_comments")
split_name = "train" if "train" in comments_ds_all else list(comments_ds_all.keys())[0]
comments_ds = comments_ds_all[split_name]
df_comments = pd.DataFrame(comments_ds).sample(n=4000, random_state=42).reset_index(drop=True)

df_comments = df_comments.rename(columns={"text": "text"})
df_comments["source"] = "comment"

# Create a simple label for visualization if toxicity exists
if "toxicity" in df_comments.columns:
    df_comments["label"] = (df_comments["toxicity"] >= 0.5).astype(int)
else:
    df_comments["label"] = 0

# Merge
df = pd.concat([df_reviews[["text","label","source"]], df_comments[["text","label","source"]]], ignore_index=True)
TEXT_COL = "text"
LABEL_COL = "label"
print("Combined dataset shape:", df.shape)
df.head()

## Step 2 — Explore the data (EDA)

We inspect sample text, length distribution, and label balance to understand what we’re working with.

In [ ]:
# 2a) EDA: Distribution by source (reviews vs comments)
# Shows how much data comes from each source
# Useful to justify "media company: reviews + comments"

import matplotlib.pyplot as plt

# Print counts
print(df["source"].value_counts())

# Bar chart
plt.figure(figsize=(6,4))
df["source"].value_counts().plot(kind="bar")
plt.title("Distribution of Text Sources (Reviews vs Comments)")
plt.xlabel("Source")
plt.ylabel("Number of Records")
plt.xticks(rotation=0)
plt.show()

In [ ]:
# 2b) Exploratory Data Analysis (EDA)
# Check label distribution
# Compute text length stats
# Visualize length distribution

df["char_len"] = df[TEXT_COL].astype(str).apply(len)
df["word_len"] = df[TEXT_COL].astype(str).apply(lambda x: len(x.split()))

print("Label distribution:\n", df[LABEL_COL].value_counts())
print("\nWord length summary:\n", df["word_len"].describe())

plt.figure()
plt.hist(df["word_len"], bins=50)
plt.title("Distribution of review lengths (in words)")
plt.xlabel("Words per text")
plt.ylabel("Count")
plt.show()

print("\nSample raw texts:")
for i in range(3):
    print(f"\n--- Sample {i+1} ---\n{df.loc[i, TEXT_COL][:400]}")

## Step 3 — Text Cleaning

We normalize text: lowercase, remove URLs, remove special characters, collapse whitespace.

In [ ]:
# 3) Text Cleaning
# Define a cleaning function
# Apply to dataset
# Show before/after examples

STOP_WORDS = set(stopwords.words("english"))

def clean_text(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r"http\S+|www\.\S+", " ", text) # remove URLs
    text = re.sub(r"[^a-z\s]", " ", text)         # keep letters/spaces only
    text = re.sub(r"\s+", " ", text).strip()      # collapse whitespace
    return text

df["clean_text"] = df[TEXT_COL].apply(clean_text)

print("Before:\n", df.loc[0, TEXT_COL][:300])
print("\nAfter:\n", df.loc[0, "clean_text"][:300])

## Step 4 — Tokenization, Stopword Removal, Lemmatization

We use spaCy for tokenization + lemmatization, and NLTK stopwords for filtering.

In [ ]:
# 4) Tokenize + Stopword Removal + Lemmatize
# Use spaCy pipeline
# Keep alphabetic tokens only
# Remove stopwords
# Lemmatize tokens

def spacy_lemmatize(text: str):
    doc = nlp(text)
    tokens = []
    for t in doc:
        if t.is_alpha:
            lemma = t.lemma_.lower().strip()
            if lemma and lemma not in STOP_WORDS:
                tokens.append(lemma)
    return tokens

df["tokens"] = df["clean_text"].apply(spacy_lemmatize)
df["processed_text"] = df["tokens"].apply(lambda toks: " ".join(toks))

print("Processed sample:\n", df.loc[0, "processed_text"][:300])
print("\nTokens sample:\n", df.loc[0, "tokens"][:40])

## Step 5 — Vocabulary Creation

We build vocabulary and show top frequent terms.

In [ ]:
# 5) Vocabulary Creation
# Build frequency distribution
# Display vocabulary size
# Show top-N frequent tokens

all_tokens = [tok for row in df["tokens"] for tok in row]
vocab_counter = Counter(all_tokens)

print("Vocabulary size:", len(vocab_counter))
print("\nTop 20 tokens:")
for w, c in vocab_counter.most_common(20):
    print(f"{w:>15} : {c}")

## Step 6 — Vectorization (Bag of Words (BoW) and TF‑IDF Vectors)

We build BoW and TF‑IDF representations using scikit-learn.

In [ ]:
# 6) BoW + TF-IDF Vectorization
# - Fit vectorizers on processed text
# - Show matrix shapes
# - Inspect feature names

corpus = df["processed_text"].tolist()

bow_vectorizer = CountVectorizer(max_features=5000)
X_bow = bow_vectorizer.fit_transform(corpus)

tfidf_vectorizer = TfidfVectorizer(max_features=5000)
X_tfidf = tfidf_vectorizer.fit_transform(corpus)

print("BoW shape:", X_bow.shape)
print("TF-IDF shape:", X_tfidf.shape)
print("\nExample features:", bow_vectorizer.get_feature_names_out()[:20])

## Step 7 — Word Embeddings (Gensim Word2Vec)

Train Word2Vec on tokenized text and inspect nearest neighbors.

In [ ]:
# 7) Word Embeddings (Word2Vec)
# Train Word2Vec on tokens
# Query similar words

sentences = df["tokens"].tolist()

w2v_model = Word2Vec(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=3,
    workers=2,
    sg=1,          # skip-gram
    epochs=10,
    seed=42
)

print("Word2Vec vocab size:", len(w2v_model.wv.index_to_key))

# Try a few example terms that often exist in review data
for query_word in ["good", "bad", "service", "food"]:
    if query_word in w2v_model.wv:
        print(f"\nMost similar to '{query_word}':")
        print(w2v_model.wv.most_similar(query_word, topn=8))

## Step 8 — Sentence Embeddings (Gensim Doc2Vec + Avg Word2Vec)

We build sentence embeddings in two ways:

1) Doc2Vec (learns document vectors directly)
2) Average Word2Vec (mean of word vectors in a text)

In [ ]:
# 8) Sentence Embeddings
# Train Doc2Vec
# Create Avg Word2Vec sentence vectors

# Doc2Vec expects TaggedDocument objects
tagged_docs = [TaggedDocument(words=toks, tags=[i]) for i, toks in enumerate(df["tokens"])]

doc2vec_model = Doc2Vec(
    documents=tagged_docs,
    vector_size=100,
    window=5,
    min_count=3,
    workers=2,
    epochs=20,
    dm=1,      # distributed memory
    seed=42
)

doc_vectors = np.vstack([doc2vec_model.dv[i] for i in range(len(df))])
print("Doc2Vec sentence embedding matrix:", doc_vectors.shape)

# Average Word2Vec vectors for each document
def avg_w2v_vector(tokens, model, dim=100):
    vecs = [model.wv[t] for t in tokens if t in model.wv]
    if len(vecs) == 0:
        return np.zeros(dim)
    return np.mean(vecs, axis=0)

avgw2v_vectors = np.vstack([avg_w2v_vector(toks, w2v_model, 100) for toks in df["tokens"]])
print("Avg Word2Vec sentence embedding matrix:", avgw2v_vectors.shape)

## Step 9 — Similarity Search (TF‑IDF + Doc2Vec)

We create a simple semantic search function:

Convert a query to TF‑IDF vector and return most similar texts
Convert a query to Doc2Vec vector and return most similar texts

In [ ]:
# 9) Similarity Search
# Implement two retrieval methods:
#   A) TF-IDF cosine similarity
#   B) Doc2Vec inferred vector cosine similarity

def search_tfidf(query, top_k=5):
    q = clean_text(query)
    q_tokens = spacy_lemmatize(q)
    q_proc = " ".join(q_tokens)
    q_vec = tfidf_vectorizer.transform([q_proc])
    sims = cosine_similarity(q_vec, X_tfidf).ravel()
    idx = np.argsort(-sims)[:top_k]
    return idx, sims[idx]

def search_doc2vec(query, top_k=5):
    q = clean_text(query)
    q_tokens = spacy_lemmatize(q)
    q_vec = doc2vec_model.infer_vector(q_tokens, epochs=30)
    sims = cosine_similarity([q_vec], doc_vectors).ravel()
    idx = np.argsort(-sims)[:top_k]
    return idx, sims[idx]

def show_results(query, idx, scores, title):
    print(f"\n=== {title} ===")
    print("Query:", query)
    for rank, (i, s) in enumerate(zip(idx, scores), start=1):
        raw = df.loc[i, TEXT_COL]
        print(f"\n#{rank} | score={s:.4f}")
        print(raw[:350])

# Demo queries
queries = [
    "excellent customer service and friendly staff",
    "the food was terrible and overpriced",
    "waiting time was too long"
]

for q in queries:
    idx, sc = search_tfidf(q, top_k=3)
    show_results(q, idx, sc, "TF-IDF Similarity Search")

    idx, sc = search_doc2vec(q, top_k=3)
    show_results(q, idx, sc, "Doc2Vec Similarity Search")

## Step 10 — Insights & Visualization
We’ll visualize Top words (bar chart), WordCloud and 2D projection of sentence embeddings (UMAP)

In [ ]:
# 10) Insights & Visualization
# Top frequent words
# WordCloud
# UMAP projection of sentence embeddings

# Top frequent words bar chart
top_words = vocab_counter.most_common(20)
words, counts = zip(*top_words)

plt.figure(figsize=(10,4))
plt.bar(words, counts)
plt.title("Top 20 most frequent tokens (after preprocessing)")
plt.xticks(rotation=45, ha="right")
plt.show()

# WordCloud
wc_text = " ".join(df["processed_text"].tolist())
wordcloud = WordCloud(width=900, height=400, background_color="white").generate(wc_text)

plt.figure(figsize=(12,5))
plt.imshow(wordcloud, interpolation="bilinear")
plt.axis("off")
plt.title("WordCloud of processed corpus")
plt.show()

# UMAP visualization (Doc2Vec embeddings)
reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, metric="cosine", random_state=42)
emb2d = reducer.fit_transform(doc_vectors)

plt.figure(figsize=(7,6))
plt.scatter(emb2d[:,0], emb2d[:,1], c=df[LABEL_COL], cmap="coolwarm", s=10, alpha=0.7)
plt.title("UMAP Projection of Doc2Vec Sentence Embeddings (colored by label)")
plt.xlabel("UMAP-1")
plt.ylabel("UMAP-2")
plt.show()

## Save Artifacts (vectorizers and models)

Saving artifacts makes your pipeline “production-like”.

In [ ]:
# =========================
# Save Artifacts
# Save vectorizers and gensim models
# =========================

import joblib

joblib.dump(bow_vectorizer, "bow_vectorizer.joblib")
joblib.dump(tfidf_vectorizer, "tfidf_vectorizer.joblib")

w2v_model.save("word2vec.model")
doc2vec_model.save("doc2vec.model")

print("Saved: vectorizers + Word2Vec + Doc2Vec")